Reverse idea - cluster onn admissions, tryna investigate the rest

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

In [5]:
PROJECT_ROOT = Path.cwd().parents[0]

DATA_PATH = PROJECT_ROOT / "Data" / "Lumiere_Country_Full_Backup.csv"
CLUSTERS_PATH = (
    PROJECT_ROOT / "outputs" / "text_clustering"
    / "results_content_only" / "clustered_films_k6.csv"
)

df_admissions = pd.read_csv(DATA_PATH)
df_clusters = pd.read_csv(CLUSTERS_PATH)

assert len(df_admissions) == len(df_clusters), "Row count mismatch!"

df = df_admissions.copy()
df["cluster"] = df_clusters["cluster"].values

print(f"Loaded {len(df)} films")

Loaded 2000 films


In [6]:
def parse_country_admissions(value):
    if pd.isna(value) or value == "Not Found":
        return {}
    try:
        return json.loads(value.replace('""', '"'))
    except (json.JSONDecodeError, TypeError):
        return {}

# Build long format first.
records = []
for idx, row in df.iterrows():
    admissions = parse_country_admissions(row["country_admissions"])
    for country, count in admissions.items():
        if country.startswith("Total"):
            continue
        records.append({
            "titleId": row["titleId"],
            "originalTitle": row["originalTitle"],
            "mainCountry": row["mainCountry"],
            "firstLanguage": row["firstLanguage"],
            "genres": row["genres"],
            "keywords": row["keywords"],
            "cluster": row["cluster"],
            "country": country,
            "admissions": count
        })

df_long = pd.DataFrame(records)

# Remove small countries and handle GB/IE overlap (same cleanup as before).
MIN_FILMS = 50
country_film_counts = df_long.groupby("country")["titleId"].nunique()
small_countries = country_film_counts[country_film_counts < MIN_FILMS].index.tolist()
df_long = df_long[~df_long["country"].isin(small_countries)].copy()

films_with_gbie = set(df_long[df_long["country"] == "GB_IE"]["titleId"])
mask_redundant = (
    df_long["country"].isin(["GB", "IE"]) &
    df_long["titleId"].isin(films_with_gbie)
)
df_long = df_long[~mask_redundant].copy()

print(f"Long format: {len(df_long)} rows")
print(f"Countries: {df_long['country'].nunique()}")
print(f"Films: {df_long['titleId'].nunique()}")

Long format: 22297 rows
Countries: 35
Films: 1840


In [7]:
# Pivot: films × countries, raw admissions, fill missing with 0.
admissions_matrix = df_long.pivot_table(
    index="titleId",
    columns="country",
    values="admissions",
    fill_value=0
)

print(f"Full matrix: {admissions_matrix.shape}")

# Convert to percentage of each film's total.
row_totals = admissions_matrix.sum(axis=1)
admissions_pct = admissions_matrix.div(row_totals, axis=0) * 100

# Filter: only films released in enough countries.
MIN_COUNTRIES = 10
countries_per_film = (admissions_matrix > 0).sum(axis=1)
valid_films = countries_per_film[countries_per_film >= MIN_COUNTRIES].index
admissions_pct_filtered = admissions_pct.loc[valid_films]

print(f"Films with {MIN_COUNTRIES}+ countries: {len(admissions_pct_filtered)} / {len(admissions_pct)}")
print(f"Countries: {admissions_pct_filtered.shape[1]}")

# Sanity check.
print(f"\nRow sums (should be ~100): min={admissions_pct_filtered.sum(axis=1).min():.1f}, max={admissions_pct_filtered.sum(axis=1).max():.1f}")

# Example film distribution.
print(f"\nExample film distribution (top 10 countries):")
example = admissions_pct_filtered.iloc[0].sort_values(ascending=False)
print(example.head(10).round(2))

Full matrix: (1840, 35)
Films with 10+ countries: 898 / 1840
Countries: 35

Row sums (should be ~100): min=100.0, max=100.0

Example film distribution (top 10 countries):
country
FR       59.51
IT       10.85
CZ        8.40
GB_IE     8.32
ES        4.45
SK        3.58
FI        1.20
CH        1.16
PL        0.83
RO        0.68
Name: tt0013442, dtype: float64


In [8]:
# Try different k values with KMeans.
K_VALUES = [4, 5, 6, 7, 8, 10]

print(f"Clustering on {admissions_pct_filtered.shape[0]} films × {admissions_pct_filtered.shape[1]} countries\n")

print(f"{'k':<5} {'Silhouette':>10} {'Inertia':>12} {'Smallest':>10} {'Largest':>10} {'Imbalance':>10}")
print("-" * 60)

results = {}
for k in K_VALUES:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(admissions_pct_filtered.values)
    
    sil = silhouette_score(admissions_pct_filtered.values, labels, sample_size=min(500, len(labels)), random_state=42)
    sizes = pd.Series(labels).value_counts().sort_index()
    imbalance = sizes.max() / sizes.min()
    
    results[k] = {"model": km, "labels": labels, "silhouette": sil}
    
    print(f"{k:<5} {sil:>10.3f} {km.inertia_:>12.0f} {sizes.min():>10} {sizes.max():>10} {imbalance:>10.1f}")

Clustering on 898 films × 35 countries

k     Silhouette      Inertia   Smallest    Largest  Imbalance
------------------------------------------------------------
4          0.233       730966        129        461        3.6
5          0.255       638310         35        443       12.7
6          0.272       560647         35        414       11.8
7          0.275       515638         34        360       10.6
8          0.256       472588         34        255        7.5
10         0.251       399750         14        213       15.2


In [9]:
# Profile k=6 and k=7.
for k in [6, 7]:
    km = results[k]["model"]
    labels = results[k]["labels"]
    
    # Attach labels to the percentage matrix.
    df_profiles = admissions_pct_filtered.copy()
    df_profiles["adm_cluster"] = labels
    
    print(f"\n{'=' * 70}")
    print(f"ADMISSIONS CLUSTERS: k={k}")
    print(f"{'=' * 70}")
    
    for c in sorted(df_profiles["adm_cluster"].unique()):
        cluster_df = df_profiles[df_profiles["adm_cluster"] == c]
        mean_pct = cluster_df.drop(columns="adm_cluster").mean().sort_values(ascending=False)
        
        print(f"\n--- Cluster {c} ({len(cluster_df)} films) ---")
        print(f"Top countries by mean % of admissions:")
        for country, pct in mean_pct.head(5).items():
            print(f"  {country:<8} {pct:>6.1f}%")
        
        # What content clusters and languages are in here?
        film_info = df[df["titleId"].isin(cluster_df.index)]
        top_langs = film_info["firstLanguage"].value_counts().head(3)
        top_origins = film_info["mainCountry"].value_counts().head(3)
        top_content = film_info["cluster"].value_counts().sort_index()
        
        print(f"Top languages: {', '.join([f'{l} ({c})' for l, c in top_langs.items()])}")
        print(f"Top origins:   {', '.join([f'{o} ({c})' for o, c in top_origins.items()])}")
        print(f"Content clusters: {', '.join([f'c{cl}:{ct}' for cl, ct in top_content.items()])}")


ADMISSIONS CLUSTERS: k=6

--- Cluster 0 (414 films) ---
Top countries by mean % of admissions:
  FR         17.9%
  IT         13.5%
  ES         12.8%
  DE         10.9%
  GB_IE       8.7%
Top languages: en (301), it (20), fr (17)
Top origins:   GB (162), FR (54), DE (40)
Content clusters: c0:58, c1:101, c2:65, c3:65, c4:80, c5:45

--- Cluster 1 (120 films) ---
Top countries by mean % of admissions:
  GB         34.0%
  FR         13.7%
  DE          9.8%
  ES          7.4%
  IT          6.7%
Top languages: en (116), fr (2), zxx (1)
Top origins:   GB (95), IE (10), FR (6)
Content clusters: c0:27, c1:19, c2:22, c3:15, c4:27, c5:10

--- Cluster 2 (131 films) ---
Top countries by mean % of admissions:
  FR         63.2%
  DE          5.3%
  ES          4.9%
  IT          4.3%
  GB_IE       3.8%
Top languages: en (64), fr (47), de (6)
Top origins:   FR (69), GB (25), DE (8)
Content clusters: c0:17, c1:27, c2:27, c3:8, c4:34, c5:18

--- Cluster 3 (40 films) ---
Top countries by mean % of 

As expected, the admissions clusters are largely driven by where the audience concentrates, and the dominant pattern is language/origin. But there are some interesting nuances.
k=7 is more informative than k=6 because it splits out an Italy-dominated cluster (cluster 4) that k=6 absorbs into the general pan-European cluster. Let me focus on k=7:
Cluster 1 (360 films) — the "pan-European" cluster. The largest group by far. Admissions spread relatively evenly across France (19%), Spain (13%), Germany (12%), GB_IE (10%), Italy (9%). These are the true travellers. Almost all English-language, mostly British origin, and content clusters are evenly represented. This is the performance profile every filmmaker wants.
Cluster 0 (131 films) — "GB-heavy." 33% of audience in GB alone. Almost entirely English-language and British-origin. The key difference from cluster 1 is that these films do well in Britain specifically but don't spread as evenly to continental Europe. Probably films with strong British cultural specificity that attract some international distribution but concentrate at home.
Cluster 5 (150 films) — "GB_IE-heavy." 44% in GB_IE. Very similar to cluster 0 but even more concentrated. This is interesting — there are TWO British-audience clusters with different degrees of concentration. Cluster 5 content clusters skew toward c2 (comedy) and c5 (adventure/family), which aligns with our earlier finding that these content types don't travel well.
Cluster 2 (128 films) — "France-dominated." 64% of audience in France. Mix of French-language and English-language films, but overwhelmingly French origin. Content clusters skew toward c4 (romance) and c2 (comedy) — the domestic-heavy content types. These are films that got international release but the audience stayed in France.
Cluster 3 (34 films) — "Spain-dominated." 64% in Spain. Same pattern — a mix of English and Spanish language, Spanish origin. Content skews toward c1 (drama) and c3 (crime thriller). Small cluster, very concentrated.
Cluster 6 (39 films) — "Germany-dominated." 54% in Germany. Mix of English and German language, German origin. Interestingly skews toward c5 (adventure/family) — German-made family films that play mostly at home.
Cluster 4 (56 films) — "Italy-leaning." 39% Italy, the rest spread across France, Spain, Germany. This is the one k=7 separates from k=6. More English-language than Italian, with content skewing toward c2 (comedy). Films that do disproportionately well in Italy without being Italian-domestic products.
The big takeaway: admissions clustering mostly re-discovers language and origin corridors, as we predicted. The clusters are essentially: "pan-European", "British domestic", "very British domestic", "French domestic", "Spanish domestic", "German domestic", and "Italy-leaning." Content cluster membership is surprisingly evenly distributed within most admissions clusters — meaning the same content types appear in every performance pattern.
Where this is genuinely useful is the contrast between cluster 1 and the rest. The 360 pan-European films versus the ~540 domestically-concentrated films is the key split. The question for your predictor becomes: what distinguishes a film that ends up in cluster 1 from one that ends up in cluster 2 or 5? We already know language and origin matter — but within English-language British films, some end up pan-European (cluster 1) and others stay GB-heavy (cluster 0/5). That difference is likely content-driven, and worth digging into.
Want to investigate that — what separates the pan-European British films from the UK-concentrated ones?

In [10]:
k = 20
km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
labels = km.fit_predict(admissions_pct_filtered.values)

sil = silhouette_score(admissions_pct_filtered.values, labels, sample_size=min(500, len(labels)), random_state=42)
sizes = pd.Series(labels).value_counts().sort_index()

print(f"k={k}, Silhouette: {sil:.3f}")
print(f"\nCluster sizes:")
print(sizes)
print(f"\nSmallest: {sizes.min()}, Largest: {sizes.max()}")

# Profile each cluster.
df_profiles = admissions_pct_filtered.copy()
df_profiles["adm_cluster"] = labels

for c in sorted(df_profiles["adm_cluster"].unique()):
    cluster_df = df_profiles[df_profiles["adm_cluster"] == c]
    mean_pct = cluster_df.drop(columns="adm_cluster").mean().sort_values(ascending=False)
    
    # Top 3 countries with their %
    top3 = ", ".join([f"{country} {pct:.0f}%" for country, pct in mean_pct.head(3).items()])
    
    # Film info.
    film_info = df[df["titleId"].isin(cluster_df.index)]
    top_lang = film_info["firstLanguage"].value_counts().index[0]
    top_origin = film_info["mainCountry"].value_counts().index[0]
    
    print(f"\nCluster {c:>2} ({len(cluster_df):>3} films) | {top3} | lang: {top_lang}, origin: {top_origin}")

k=20, Silhouette: 0.201

Cluster sizes:
0     108
1      71
2      42
3      25
4      89
5      90
6      36
7      22
8      16
9      55
10     10
11     39
12     62
13     58
14     18
15      9
16     90
17     36
18     12
19     10
Name: count, dtype: int64

Smallest: 9, Largest: 108

Cluster  0 (108 films) | GB_IE 22%, FR 18%, DE 15% | lang: en, origin: GB

Cluster  1 ( 71 films) | FR 61%, DE 6%, IT 6% | lang: en, origin: FR

Cluster  2 ( 42 films) | GB 51%, DE 8%, FR 8% | lang: en, origin: GB

Cluster  3 ( 25 films) | IT 51%, FR 8%, ES 7% | lang: en, origin: IT

Cluster  4 ( 89 films) | GB 23%, FR 19%, DE 10% | lang: en, origin: GB

Cluster  5 ( 90 films) | GB_IE 39%, FR 12%, ES 10% | lang: en, origin: GB

Cluster  6 ( 36 films) | GB_IE 64%, FR 8%, ES 7% | lang: en, origin: GB

Cluster  7 ( 22 films) | ES 44%, IT 11%, FR 8% | lang: en, origin: GB

Cluster  8 ( 16 films) | DE 71%, CH 4%, FR 4% | lang: de, origin: DE

Cluster  9 ( 55 films) | DE 11%, IT 11%, FR 9% | lang: en, o

Now we're getting somewhere. With 20 clusters, the language/origin groupings break apart and reveal genuine geographic performance patterns. Let me group them by what they tell us:

**The domestic-dominated clusters (films that stay home):**
- Cluster 17 (36 films) — **French ultra-domestic.** 82% France. French-language, French origin. These barely register anywhere else.
- Cluster 14 (18 films) — **Spanish ultra-domestic.** 78% Spain. Same pattern.
- Cluster 8 (16 films) — **German ultra-domestic.** 71% Germany. Same pattern.
- Cluster 10 (10 films) — **Danish domestic.** 58% Denmark. Small but distinct.

**The UK spectrum — this is the most interesting part.** British films split into five different performance profiles:
- Cluster 6 (36 films) — **UK-locked.** 64% GB_IE. Hardly travels at all.
- Cluster 2 (42 films) — **GB-heavy.** 51% GB (not GB_IE). Slightly different distribution channel.
- Cluster 5 (90 films) — **UK-leaning but some spread.** 39% GB_IE, with some reaching Spain and France.
- Cluster 4 (89 films) — **UK with continental reach.** 23% GB, 19% France, 10% Germany. Starting to travel.
- Cluster 0 (108 films) — **True pan-European from UK.** 22% GB_IE, 18% France, 15% Germany. The most evenly spread British cluster.

This gradient from cluster 6 → 0 is exactly what your predictor needs to understand. Same language, same origin, but wildly different distribution outcomes. The content driving a film from cluster 6 to cluster 0 is the actionable insight.

**The continental corridors — genuinely new findings:**
- Cluster 19 (10 films) — **Nordic cluster.** 20% Denmark, 16% Sweden, 14% Norway. Swedish-language, Swedish origin. These are films that circulate within Scandinavia. Small but distinct.
- Cluster 18 (12 films) — **Benelux cluster.** 32% Netherlands, 16% France, 12% Belgium. English-language but performing in the Low Countries. An unexpected corridor.
- Cluster 15 (9 films) — **Polish cluster.** 51% Poland, 14% France. English-language but Polish origin. Very niche.
- Cluster 12 (62 films) — **Southern European.** 22% France, 21% Italy, 15% Spain. Films that play well in the Mediterranean markets but not in Northern Europe.
- Cluster 9 (55 films) — **The true universals.** 11% Germany, 11% Italy, 9% France — the most evenly spread cluster. No single market dominates. These are the films every market gets roughly equal share of.

**The hybrid clusters:**
- Cluster 1 (71 films) and 16 (90 films) — **French-heavy but travelling.** 61% and 36% France respectively. French origin but with some international reach — cluster 16 is the "French film that partly travels" while cluster 1 is "French film that barely travels."
- Cluster 7 (22 films) — **Spain-leaning international.** 44% Spain but English-language, British origin. British films that happen to resonate in Spain.
- Cluster 3 (25 films) — **Italy-leaning.** 51% Italy, English-language. Films that over-index in Italy.
- Cluster 13 (58 films) — **Spain-leaning pan-European.** 25% Spain, 16% GB_IE, 15% Germany. Broad but with a Spanish tilt.

**The key insight for your project:** cluster 9 (the true universals) and cluster 19 (the Nordic corridor) are the most interesting for follow-up. Cluster 9 contains 55 films that managed to spread almost perfectly evenly across Europe — what do they have in common? And cluster 19 shows a genuine regional corridor that isn't just about language. Want to dig into the content of either of these?